# VMAS

> VMAS scenarios (vectorized multi-robot simulator) as stable-marl environments, with vector observations.

In [ ]:
#| default_exp envs.vmas.env

In [ ]:
#| hide
from nbdev.showdoc import *

[VMAS](https://github.com/proroklab/VectorizedMultiAgentSimulator) (Bettini et al.): 2D multi-robot
scenarios (navigation, transport, balance, flocking, football, ... and the MPE tasks) simulated with
PyTorch. VMAS is GPL-3.0, so it is not vendored: it is an optional dependency
(`pip install 'stable-marl[vmas]'`), imported only when a VMAS env is created.

`VmasEnv` runs one VMAS scenario (VMAS's `num_envs=1`; a `World`'s pool holds several) as a stable-marl
`MultiAgentEnv`, observations being vectors (no rendering):

| key | per agent | content |
|---|---|---|
| `vector` | `(dim,)` float32 | VMAS's observation (zero-padded to the largest agent's, for scenarios with agents of different kinds) |
| the scenario's infos | | e.g. `pos_rew`, `final_rew`, `agent_collisions` for navigation |

Actions are continuous (VMAS's default; the actions of agents with fewer dimensions are padded with
dimensions bounded to 0, dropped before VMAS) or discrete (`continuous_actions=False`). VMAS's end of
episode applies to every agent: termination (the scenario's `done`) or truncation (`max_steps`).
`state()` is the position and velocity of every entity (agents and landmarks).

In [ ]:
#| export
from __future__ import annotations

from typing import Any

import numpy as np
from gymnasium import spaces

from stable_marl.envs.base import MultiAgentEnv
from stable_marl.types import AgentID

In [ ]:
#| export
def _gymnasium_space(space) -> spaces.Space:
    "A gymnasium copy of one of VMAS's gym spaces."
    kind = type(space).__name__
    if kind == 'Box':
        return spaces.Box(np.asarray(space.low, np.float32), np.asarray(space.high, np.float32), dtype=np.float32)
    if kind == 'Discrete':
        return spaces.Discrete(int(space.n))
    if kind == 'MultiDiscrete':
        return spaces.MultiDiscrete(np.asarray(space.nvec))
    raise TypeError(f"unsupported VMAS space {space}")


def _numpy(value):
    value = value.detach().cpu().numpy()[0] if hasattr(value, 'detach') else np.asarray(value)[0]
    return value.item() if np.ndim(value) == 0 else value


class VmasEnv(MultiAgentEnv):
    "The VMAS `scenario` (one of `vmas.scenarios` / `vmas.mpe_scenarios`) as a stable-marl env."
    def __init__(
        self,
        scenario: str, # VMAS scenario, e.g. 'navigation', 'transport', 'simple_spread'
        continuous_actions: bool = True, # Continuous (VMAS's default) or discrete actions
        max_steps: int = 100, # Step limit (truncation); VMAS's default is none
        device: str = 'cpu', # Torch device of the simulation
        **scenario_kwargs # The scenario's arguments, e.g. `n_agents=4`
    ):
        import vmas
        self.scenario = scenario
        self._env = vmas.make_env(scenario, num_envs=1, device=device, continuous_actions=continuous_actions,
                                  dict_spaces=True, terminated_truncated=True, max_steps=max_steps, **scenario_kwargs)
        self.agent_names = [agent.name for agent in self._env.agents]
        self.num_agents = len(self.agent_names)
        self.continuous_actions = continuous_actions
        obs = [_gymnasium_space(self._env.observation_space[n]) for n in self.agent_names]
        acts = [_gymnasium_space(self._env.action_space[n]) for n in self.agent_names]
        self.obs_dims = [int(np.prod(s.shape)) for s in obs]                # each agent's true sizes
        self.action_dims = [int(np.prod(s.shape)) for s in acts]
        D = max(self.obs_dims)
        self.observation_space = spaces.Dict({a: spaces.Dict({'vector': spaces.Box(-np.inf, np.inf, (D,), np.float32)})
                                              for a in range(self.num_agents)})
        if continuous_actions:
            K = max(self.action_dims)
            pad = lambda x, d: np.concatenate([x.reshape(-1), np.zeros(K - d, np.float32)])
            self.action_space = spaces.Dict({a: spaces.Box(pad(s.low, d), pad(s.high, d), dtype=np.float32)
                                             for a, (s, d) in enumerate(zip(acts, self.action_dims))})
        else:
            if len({str(s) for s in acts}) > 1:
                raise ValueError(f"{scenario}: the agents' discrete action spaces differ ({acts}); use continuous actions")
            self.action_space = spaces.Dict({a: s for a, s in enumerate(acts)})

    @property
    def noop_action(self):
        "Zero force (continuous), or VMAS's action 0 (no movement)."
        return np.zeros(self.action_space[0].shape, np.float32) if self.continuous_actions else 0

    @property
    def state_space(self) -> spaces.Space:
        "Position and velocity of every entity: (num_entities, 2 * dim)."
        n, d = len(self._env.world.entities), self._env.world.dim_p
        return spaces.Box(-np.inf, np.inf, (n, 2 * d), np.float32)

    def state(self) -> np.ndarray:
        return np.stack([np.concatenate([_numpy(e.state.pos), _numpy(e.state.vel)])
                         for e in self._env.world.entities]).astype(np.float32)

    def _observations(self, obs) -> dict[AgentID, dict]:
        D = max(self.obs_dims)
        out = {}
        for a, name in enumerate(self.agent_names):
            v = np.asarray(_numpy(obs[name]), np.float32).reshape(-1)
            out[a] = {'vector': np.concatenate([v, np.zeros(D - len(v), np.float32)])}
        return out

    def _infos(self, info) -> dict[AgentID, dict]:
        return {a: {k: _numpy(v) for k, v in info[name].items()} for a, name in enumerate(self.agent_names)}

    def reset(self, seed: int | None = None, options: dict | None = None):
        super().reset(seed=seed)
        obs, info = self._env.reset(seed=seed, return_info=True)
        return self._observations(obs), self._infos(info)

    def step(self, actions: dict[AgentID, Any]):
        import torch
        act = {}
        for a, name in enumerate(self.agent_names):
            x = np.asarray(actions[a])
            if self.continuous_actions:
                act[name] = torch.as_tensor(x.reshape(-1)[:self.action_dims[a]], dtype=torch.float32)[None]
            else:
                act[name] = torch.as_tensor(x.astype(np.int64)).reshape(1, *self.action_space[a].shape)
        obs, rewards, terminated, truncated, info = self._env.step(act)
        term, trunc = bool(terminated[0]), bool(truncated[0])
        agents = range(self.num_agents)
        reward = lambda r: float(np.asarray(_numpy(r)).reshape(-1)[0])   # (1,) or (1, 1) depending on the scenario
        return (self._observations(obs), {a: reward(rewards[n]) for a, n in enumerate(self.agent_names)},
                {a: term for a in agents}, {a: trunc for a in agents}, self._infos(info))

### Tests

In [ ]:
from fastcore.test import test_fail
import torch

env = VmasEnv('navigation', n_agents=2, max_steps=10)
obs, infos = env.reset(seed=0)
assert set(obs) == {0, 1} and obs[0]['vector'].shape == (18,) and env.observation_space[0].contains(obs[0])
assert set(infos[0]) == {'pos_rew', 'final_rew', 'agent_collisions'}
assert env.state().shape == env.state_space.shape == (4, 4)                    # 2 goals + 2 agents
assert np.array_equal(VmasEnv('navigation', n_agents=2).reset(seed=0)[0][1]['vector'], obs[1]['vector'])   # seeded
for t in range(10):
    obs, rew, term, trunc, info = env.step({a: env.action_space[a].sample() for a in range(2)})
assert all(trunc.values()) and set(rew) == {0, 1} and set(info[1]) == set(infos[1])

# discrete actions, and VMAS's no-op
env = VmasEnv('navigation', continuous_actions=False, n_agents=2)
env.reset(seed=0)
assert env.action_space[0] == spaces.Discrete(9) and env.noop_action == 0
before = env.state()
env.step({0: 0, 1: 0})
assert np.allclose(env.state()[2:, :2], before[2:, :2], atol=1e-6)            # agents at rest stay put

# agents of different kinds: observations zero-padded, actions padded with dimensions bounded to 0
env = VmasEnv('simple_speaker_listener')
obs, _ = env.reset(seed=0)
assert env.obs_dims == [3, 11] and obs[0]['vector'].shape == (11,) and (obs[0]['vector'][3:] == 0).all()
assert env.action_dims == [5, 2] and env.action_space[1].shape == (5,) and (env.action_space[1].high[2:] == 0).all()
env.step({a: env.action_space[a].sample() for a in range(2)})
test_fail(lambda: VmasEnv('simple_speaker_listener', continuous_actions=False), contains='differ')

# every scenario runs in a World: datasets, evaluation
import tempfile
from stable_marl import World, RandomPolicy
from stable_marl.data import HDF5Dataset
import vmas
skip = {'road_traffic'}   # slow to load (a road map with 20 agents)
with tempfile.TemporaryDirectory() as tmp:
    for scenario in [*vmas.scenarios, *vmas.mpe_scenarios]:
        if scenario in skip:
            continue
        world = World(f'VMAS-{"".join(p.capitalize() for p in scenario.split("_"))}-v0', num_envs=2, max_steps=8)
        world.set_policy(RandomPolicy(seed=0))
        world.collect(f'{tmp}/{scenario}.h5', episodes=2, seed=0, progress=False)
        ds = HDF5Dataset(path=f'{tmp}/{scenario}.h5', num_steps=3)
        A = world.num_agents
        assert ds[0]['vector'].shape[:2] == (3, A) and ds[0]['action'].shape[0] == 3, scenario
        assert len(world.evaluate(episodes=2, seed=1)['episode_returns']) == 2

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()